In [1]:
!pip install numpy==1.26.4
!pip install scikit-learn
!pip install scikit-learn-extra
!pip install rdkit

In [2]:
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs
from sklearn_extra.cluster import KMedoids

In [15]:
def read_smiles(file_path: str) -> list[Chem.Mol]:
    # Read SMILES strings from file and convert to molecules
    mols = []
    with open(file_path, 'r') as f:
        for line in f:
            # Split on colon and take the SMILES part
            smiles = line.strip().split(': ')[1]
            mol = Chem.MolFromSmiles(smiles)
            if mol is not None:  # Only add valid molecules
                mols.append(mol)
            else:
                print(f"Invalid smiles: {line.strip()}")
    return mols

mols = read_smiles('smiles.txt')

In [16]:
def get_fps(mols: list[Chem.Mol]) -> list[np.ndarray]:
    fps  = [AllChem.GetMorganFingerprintAsBitVect(m, 3, nBits=2048) for m in mols]
    return fps

fps = get_fps(mols)

[22:21:53] DEPRECATION WARNING: please use MorganGenerator
[22:21:53] DEPRECATION WARNING: please use MorganGenerator
[22:21:53] DEPRECATION WARNING: please use MorganGenerator
[22:21:53] DEPRECATION WARNING: please use MorganGenerator
[22:21:53] DEPRECATION WARNING: please use MorganGenerator
[22:21:53] DEPRECATION WARNING: please use MorganGenerator
[22:21:53] DEPRECATION WARNING: please use MorganGenerator
[22:21:53] DEPRECATION WARNING: please use MorganGenerator
[22:21:53] DEPRECATION WARNING: please use MorganGenerator
[22:21:53] DEPRECATION WARNING: please use MorganGenerator
[22:21:53] DEPRECATION WARNING: please use MorganGenerator
[22:21:53] DEPRECATION WARNING: please use MorganGenerator
[22:21:53] DEPRECATION WARNING: please use MorganGenerator
[22:21:53] DEPRECATION WARNING: please use MorganGenerator
[22:21:53] DEPRECATION WARNING: please use MorganGenerator
[22:21:53] DEPRECATION WARNING: please use MorganGenerator
[22:21:53] DEPRECATION WARNING: please use MorganGenerat

In [17]:
def tanimoto(i, j): # tanimoto similarity captures how similar two molecules are
    return DataStructs.TanimotoSimilarity(fps[i], fps[j])

def ligand_distance_matrix(fps: list[np.ndarray]) -> np.ndarray:
    d_lig = np.array([[1.0 - tanimoto(i,j) for j in range(len(fps))] for i in range(len(fps))])
    return d_lig

d_lig = ligand_distance_matrix(fps)

In [18]:
def pair_distance_matrix(d_lig: np.ndarray) -> tuple[list[tuple[int, int]], np.ndarray]:
    pairs = [(i,j) for i in range(len(d_lig)) for j in range(i+1,len(d_lig))]
    P = len(pairs)
    D_pair = np.zeros((P,P))
    for a,(i,j) in enumerate(pairs):
        for b,(p,q) in enumerate(pairs[a:], start=a):
            d1 = d_lig[i,p] + d_lig[j,q]
            d2 = d_lig[i,q] + d_lig[j,p]
            D_pair[a,b] = D_pair[b,a] = 0.5*min(d1,d2)
    return pairs, D_pair

pairs, D_pair = pair_distance_matrix(d_lig)

In [ ]:
def choose_mol_pairs(pairs: list[tuple[int, int]], D_pair: np.ndarray) -> list[tuple[int, int]]:
    K = 10
    # random state is set to 42 to ensure reproducibility later
    med = KMedoids(n_clusters=K, metric='precomputed', init='k-medoids++', random_state=42).fit(D_pair)
    chosen_pairs = [pairs[idx] for idx in med.medoid_indices_]
    chosen_smiles_pairs = [(Chem.MolToSmiles(mols[i]), Chem.MolToSmiles(mols[j])) for i, j in chosen_pairs]
    return chosen_pairs, chosen_smiles_pairs

chosen_pairs, chosen_smiles_pairs = choose_mol_pairs(pairs, D_pair)
chosen_pairs

[(2, 9),
 (13, 14),
 (0, 7),
 (5, 16),
 (0, 1),
 (1, 6),
 (3, 10),
 (7, 11),
 (4, 15),
 (12, 15)]

In [ ]:
chosen_smiles_pairs

[('Nc1cc(C(=O)O)cc(C(=O)O)c1', 'O=C(O)c1cc(CBr)cc(C(=O)O)c1'),
 ('O=C(O)c1cc(Oc2ccccc2)cc(C(=O)O)c1', 'CCOc1cc(C(=O)O)cc(C(=O)O)c1'),
 ('Cc1cc(C(=O)O)cc(C(=O)O)c1', 'O=C(O)c1cc(C(=O)O)c(O)cc1O'),
 ('O=C(O)c1cc(C(=O)O)cc(S(=O)(=O)[O-])c1',
  'O=C(O)c1cc(OCc2ccncc2)cc(C(=O)O)c1'),
 ('Cc1cc(C(=O)O)cc(C(=O)O)c1', 'O=C(O)c1cc(O)cc(C(=O)O)c1'),
 ('O=C(O)c1cc(O)cc(C(=O)O)c1', 'Cc1cc(C)c(C(=O)O)cc1C(=O)O'),
 ('N#Cc1cc(C(=O)O)cc(C(=O)O)c1', 'CC(C)(C)C(=O)Nc1cc(C(=O)O)cc(C(=O)O)c1'),
 ('O=C(O)c1cc(C(=O)O)c(O)cc1O', 'O=C(O)c1cc(C(=O)O)cc(-c2ccccc2)c1'),
 ('O=C(O)c1cc(Br)cc(C(=O)O)c1', 'CCCCCOc1cc(C(=O)O)cc(C(=O)O)c1'),
 ('O=C(O)c1cc(C(=O)O)cc(N(c2ccccc2)c2ccccc2)c1',
  'CCCCCOc1cc(C(=O)O)cc(C(=O)O)c1')]